# Model A — Segmentation Model

## 현재 상태

Segmentation GT Rasterization은 전체 **8,788장**에서 검증 완료되었다.

현재 태건 쪽 Segmentation Task에서는 다음 체크포인트까지 완료된 상태다.

- Shared Encoder → Segmentation Head Input Contract 확인 ✅
- Progressive Multi-scale Segmentation Head baseline 설계 ✅
- `Conv → GroupNorm → GELU` DecoderBlock 적용 ✅
- Dummy Forward `[B, 3, 448, 768]` 검증 ✅
- `src/segmentation/head.py` 모듈화 및 src import 회귀 검증 ✅
- `BCEWithLogitsLoss + Soft Dice` Loss baseline 설계 ✅
- sample × channel Dice / non-empty GT 처리 검증 ✅
- Full-size Loss Forward / Backward / Gradient 검증 ✅
- `src/segmentation/loss.py` 모듈화 및 src import 회귀 검증 ✅

### 다음 큰 단계

`Dataset / DataLoader → 실제 Segmentation GT → Prediction ↔ GT Loss` 연결을 진행한다.

> **중요:** 아직 주한의 Shared Encoder와 실제 RGB를 연결해 학습하는 단계는 아니다.  
> 먼저 태건 쪽의 Dataset / GT / Loss 연결을 안정적으로 만든 뒤 실제 Encoder와 연결한다.

## 전체 Model A에서 태건 작업 위치

```text
RGB
 ↓
Shared Encoder                          ← 주한
 ↓
C1 / C2 / C3 / C4
 ├───────────────────────────┐
 ↓                           ↓
Segmentation Head            Vector Head
← 태건                       ← 주한
 ↓
[B, 3, 448, 768] logits
 ↓
Segmentation Loss
 ↓
Training / Validation
```

태건이 맡은 **Segmentation Task** 안에는 다음 작업이 포함된다.

```text
Segmentation Task
├─ GT Rasterization                  ✅
├─ Segmentation Head                 ✅ baseline
├─ Segmentation Loss                 ✅ baseline
├─ Dataset / DataLoader              ← 다음
├─ Prediction ↔ GT 연결
├─ Training
├─ Validation / Metric
└─ Prediction QC
```

`Segmentation Head`는 Segmentation Task 전체가 아니라, Encoder feature를 받아 segmentation logits를 만드는 **모델 부품 하나**다.

# 1. Shared Encoder → Segmentation Head Input Contract

## 역할 구분

Shared Encoder 구현 및 최종 Multi-task 통합은 **이주한 담당**이다.

태건 Segmentation에서는 별도의 Encoder를 다시 구현하지 않는다.  
Segmentation Head는 주한 Shared Encoder가 반환하는 multi-scale feature를 그대로 입력으로 사용한다.

## Shared Encoder

Backbone:

- `ConvNeXt-Tiny`
- ImageNet pretrained
- `features_only=True`
- `out_indices=(0, 1, 2, 3)`

실제 Model A 입력:

```text
[B, 3, 448, 768]
```

실제 Encoder 출력:

| Feature | Shape | Stride | 쉬운 의미 |
|---|---|---:|---|
| `c1` | `[B, 96, 112, 192]` | 4 | 가장 세밀한 위치 정보가 상대적으로 많이 남음 |
| `c2` | `[B, 192, 56, 96]` | 8 | C1보다 작지만 아직 세밀한 정보가 많음 |
| `c3` | `[B, 384, 28, 48]` | 16 | 더 깊은 문맥 정보 |
| `c4` | `[B, 768, 14, 24]` | 32 | 가장 깊고 coarse한 문맥 정보 |

Encoder 반환 형식:

```python
{
    "c1": c1,
    "c2": c2,
    "c3": c3,
    "c4": c4,
}
```

### `[B, C, H, W]` 뜻

- `B`: Batch size
- `C`: Feature channel 수
- `H`: Feature map 높이
- `W`: Feature map 너비

## Segmentation Head Output Contract

최종 출력:

```text
[B, 3, 448, 768] logits
```

채널 의미:

- channel 0 → `traffic_lane`
- channel 1 → `stop_line`
- channel 2 → `crosswalk`

세 채널은 서로 배타적인 class ID가 아니다.  
각 채널은 **독립적인 binary logit**이다.

따라서 한 픽셀에서 세 채널이 모두 negative라면 background로 해석한다.

> Head에서는 `sigmoid`를 적용하지 않는다.  
> `BCEWithLogitsLoss`는 raw logits를 직접 받고, Soft Dice 계산 내부에서만 `sigmoid`를 사용한다.

# 2. Segmentation Head Design

## 왜 Multi-scale feature를 합치는가?

우리 task는 서로 특성이 다르다.

### traffic_lane
- 얇고 긴 구조
- 정확한 위치가 중요
- 도로 전체 lane-flow 문맥도 필요

### stop_line
- 매우 얇은 선 구조
- 위치 정확도가 특히 중요

### crosswalk
- 비교적 넓은 polygon 영역
- local boundary와 scene-level context가 모두 중요

그래서 깊은 feature의 문맥 정보와 얕은 feature의 세밀한 위치 정보를 함께 쓰는 방향을 baseline으로 잡았다.

## 2-1. Decoder 후보와 현재 결정

검토한 후보:

### Candidate A — FPN-style Decoder
Top-down feature pyramid 방식으로 multi-scale feature를 결합한다.

### Candidate B — Progressive Decoder
가장 깊은 `C4`에서 시작해 `C3 → C2 → C1` 순서로 단계적으로 feature를 결합한다.

### 현재 Baseline

첫 baseline은 **Progressive Multi-scale Decoder**다.

```text
C4 + C3 → D3
D3 + C2 → D2
D2 + C1 → D1
D1 → 3-channel logits
→ [B, 3, 448, 768]
```

`D3`, `D2`, `D1`은 표준 모델 이름이 아니다.

- `D` = Decoder feature
- 숫자 = 현재 결합이 끝난 Encoder scale을 알아보기 쉽게 붙인 이름

즉 `D3`는 **C4와 C3 정보를 합쳐 만든 중간 feature map**이다.

## 2-2. 현재 Head Baseline 설정

- Encoder stages: `C1 ~ C4`
- decoder channels: `256`
- upsampling: `bilinear interpolation`
- feature fusion: `concatenation`
- DecoderBlock: `3×3 Conv → GroupNorm → GELU`
- output: `[B, 3, 448, 768]`

이 값들은 **첫 baseline**이며 최종 최적값으로 고정한 것이 아니다.

향후 학습 결과에 따라:
- stage 조합
- decoder channel
- fusion 방식
- norm / activation
등을 ablation으로 비교할 수 있다.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.13.0+cu126
CUDA build: 12.6
CUDA available: True


## 2-3. Dummy Feature를 사용하는 이유

현재 Head를 검증할 때 실제 Encoder를 연결하지 않고 `torch.randn()`으로 feature를 만든다.

이 값은 **실제 의미 있는 Encoder feature가 아니라 shape만 똑같은 가짜 테스트 데이터**다.

이렇게 하면 Head 자체에 문제가 있는지 독립적으로 확인할 수 있다.

```text
실제 Encoder C2
→ 이미지를 보고 만든 의미 있는 feature

Dummy C2
→ 값은 랜덤이지만 shape만 실제 C2와 같음
```

In [2]:
# ============================================================
# Shared Encoder 실제 output shape 기준 Dummy Features
# ============================================================

BATCH_SIZE = 2
DECODER_CHANNELS = 256

c1 = torch.randn(BATCH_SIZE, 96, 112, 192)
c2 = torch.randn(BATCH_SIZE, 192, 56, 96)
c3 = torch.randn(BATCH_SIZE, 384, 28, 48)
c4 = torch.randn(BATCH_SIZE, 768, 14, 24)

print("C1:", c1.shape)
print("C2:", c2.shape)
print("C3:", c3.shape)
print("C4:", c4.shape)

C1: torch.Size([2, 96, 112, 192])
C2: torch.Size([2, 192, 56, 96])
C3: torch.Size([2, 384, 28, 48])
C4: torch.Size([2, 768, 14, 24])


## 2-4. DecoderBlock 이해

Fusion 과정에서 feature를 `concat`하면 두 정보를 단순히 옆으로 붙인 상태다.

예:

```text
C4 projected: 256 channels
C3 projected: 256 channels
           ↓ concat
          512 channels
```

이 512채널을 실제로 학습해 섞고 다시 256채널로 정리하는 부분이 DecoderBlock이다.

```text
Concat
 ↓
3×3 Conv        ← feature를 학습해서 섞고 512 → 256으로 정리
 ↓
GroupNorm       ← feature 값 분포를 정돈
 ↓
GELU            ← 비선형성을 추가
 ↓
Decoder feature
```

GroupNorm과 GELU는 spatial shape `(H, W)`를 바꾸지 않는다.

In [3]:
# ============================================================
# Decoder Block
# Conv → GroupNorm → GELU
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        num_groups=32,
    ):
        super().__init__()

        if out_channels % num_groups != 0:
            raise ValueError(
                f"out_channels({out_channels}) must be divisible "
                f"by num_groups({num_groups})."
            )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(
                num_groups=num_groups,
                num_channels=out_channels,
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)

## 2-5. Progressive Decoder Shape 흐름

### Step 1 — C4 + C3 → D3

```text
C4 [B, 768, 14, 24]
 ↓ 1×1 Conv
[B, 256, 14, 24]
 ↓ upsample
[B, 256, 28, 48]

C3 [B, 384, 28, 48]
 ↓ 1×1 Conv
[B, 256, 28, 48]

concat
 ↓
[B, 512, 28, 48]

DecoderBlock
 ↓
D3 [B, 256, 28, 48]
```

`upsample`은 쉽게 말해 **다음 feature와 합칠 수 있도록 feature map을 확대하는 것**이다.
새로운 정보가 생긴다기보다 기존 정보를 더 큰 격자에 맞춰 펼친다.

In [4]:
# ============================================================
# C4 + C3 → D3
# ============================================================

c4_projection = nn.Conv2d(768, DECODER_CHANNELS, kernel_size=1)
c3_projection = nn.Conv2d(384, DECODER_CHANNELS, kernel_size=1)

c4_projected = c4_projection(c4)
c3_projected = c3_projection(c3)

c4_upsampled = F.interpolate(
    c4_projected,
    size=c3_projected.shape[-2:],
    mode="bilinear",
    align_corners=False,
)

fused_c43 = torch.cat(
    [c4_upsampled, c3_projected],
    dim=1,
)

c43_fusion = DecoderBlock(
    in_channels=DECODER_CHANNELS * 2,
    out_channels=DECODER_CHANNELS,
)

d3 = c43_fusion(fused_c43)

print("C4 projected :", c4_projected.shape)
print("C3 projected :", c3_projected.shape)
print("C4 upsampled :", c4_upsampled.shape)
print("C4 + C3 fused:", fused_c43.shape)
print("D3            :", d3.shape)

C4 projected : torch.Size([2, 256, 14, 24])
C3 projected : torch.Size([2, 256, 28, 48])
C4 upsampled : torch.Size([2, 256, 28, 48])
C4 + C3 fused: torch.Size([2, 512, 28, 48])
D3            : torch.Size([2, 256, 28, 48])


### Step 2 — D3 + C2 → D2

D3에는 이미 C4와 C3 정보가 들어 있다.  
여기에 더 높은 해상도의 C2 정보를 추가한다.

```text
D3 [B, 256, 28, 48]
 ↓ upsample
[B, 256, 56, 96]

C2 [B, 192, 56, 96]
 ↓ 1×1 Conv
[B, 256, 56, 96]

concat → DecoderBlock
 ↓
D2 [B, 256, 56, 96]
```

개념적으로:

```text
D3 = C4 + C3 정보
D2 = C4 + C3 + C2 정보
```

In [5]:
# ============================================================
# D3 + C2 → D2
# ============================================================

c2_projection = nn.Conv2d(192, DECODER_CHANNELS, kernel_size=1)
c2_projected = c2_projection(c2)

d3_upsampled = F.interpolate(
    d3,
    size=c2_projected.shape[-2:],
    mode="bilinear",
    align_corners=False,
)

fused_d3c2 = torch.cat(
    [d3_upsampled, c2_projected],
    dim=1,
)

d3c2_fusion = DecoderBlock(
    in_channels=DECODER_CHANNELS * 2,
    out_channels=DECODER_CHANNELS,
)

d2 = d3c2_fusion(fused_d3c2)

print("C2 projected :", c2_projected.shape)
print("D3 upsampled :", d3_upsampled.shape)
print("D3 + C2 fused:", fused_d3c2.shape)
print("D2            :", d2.shape)

C2 projected : torch.Size([2, 256, 56, 96])
D3 upsampled : torch.Size([2, 256, 56, 96])
D3 + C2 fused: torch.Size([2, 512, 56, 96])
D2            : torch.Size([2, 256, 56, 96])


### Step 3 — D2 + C1 → D1

같은 원리로 가장 높은 Encoder 해상도의 C1 정보를 추가한다.

```text
D2 [B, 256, 56, 96]
 ↓ upsample
[B, 256, 112, 192]

C1 [B, 96, 112, 192]
 ↓ 1×1 Conv
[B, 256, 112, 192]

concat → DecoderBlock
 ↓
D1 [B, 256, 112, 192]
```

개념적으로:

```text
D1 = C4 + C3 + C2 + C1 정보를 단계적으로 결합한 Decoder feature
```

In [6]:
# ============================================================
# D2 + C1 → D1
# ============================================================

c1_projection = nn.Conv2d(96, DECODER_CHANNELS, kernel_size=1)
c1_projected = c1_projection(c1)

d2_upsampled = F.interpolate(
    d2,
    size=c1_projected.shape[-2:],
    mode="bilinear",
    align_corners=False,
)

fused_d2c1 = torch.cat(
    [d2_upsampled, c1_projected],
    dim=1,
)

d2c1_fusion = DecoderBlock(
    in_channels=DECODER_CHANNELS * 2,
    out_channels=DECODER_CHANNELS,
)

d1 = d2c1_fusion(fused_d2c1)

print("C1 projected :", c1_projected.shape)
print("D2 upsampled :", d2_upsampled.shape)
print("D2 + C1 fused:", fused_d2c1.shape)
print("D1            :", d1.shape)

C1 projected : torch.Size([2, 256, 112, 192])
D2 upsampled : torch.Size([2, 256, 112, 192])
D2 + C1 fused: torch.Size([2, 512, 112, 192])
D1            : torch.Size([2, 256, 112, 192])


### Step 4 — D1 → 최종 Segmentation logits

D1은 아직 256채널짜리 Decoder feature다.

마지막 `1×1 Conv`로 256채널을 세 개의 segmentation 채널로 바꾼 뒤,
GT 크기인 `448×768`까지 확대한다.

```text
D1
[B, 256, 112, 192]

 ↓ 1×1 Conv

[B, 3, 112, 192]

 ↓ bilinear upsample

[B, 3, 448, 768]
```

이 세 채널 값은 0/1 mask가 아니라 **raw logits**다.

In [7]:
# ============================================================
# D1 → Final Segmentation Logits
# ============================================================

seg_classifier = nn.Conv2d(
    in_channels=DECODER_CHANNELS,
    out_channels=3,
    kernel_size=1,
)

seg_logits_lowres = seg_classifier(d1)

seg_logits = F.interpolate(
    seg_logits_lowres,
    size=(448, 768),
    mode="bilinear",
    align_corners=False,
)

print("Low-res logits:", seg_logits_lowres.shape)
print("Final logits  :", seg_logits.shape)

Low-res logits: torch.Size([2, 3, 112, 192])
Final logits  : torch.Size([2, 3, 448, 768])


# 3. Final SegmentationHead Baseline

위에서 단계별로 검증한 코드를 실제로 재사용하기 위해
하나의 `nn.Module`로 묶는다.

최종 구조:

```text
features = {
    c1, c2, c3, c4
}
        ↓
SegmentationHead
        ↓
[B, 3, output_H, output_W]
```

설계 원칙:

- `encoder_channels`를 인자로 받아 Encoder channel을 한 곳에서 관리
- `decoder_channels`도 configurable
- 출력 크기는 Head 안에 hardcoding하지 않고 `output_size`를 외부에서 전달
- 필요한 feature key가 빠지면 바로 오류를 발생시켜 디버깅을 쉽게 함

In [8]:
# ============================================================
# Progressive Segmentation Head
# ============================================================

class SegmentationHead(nn.Module):

    def __init__(
        self,
        encoder_channels=(96, 192, 384, 768),
        decoder_channels=256,
        num_classes=3,
        num_groups=32,
    ):
        super().__init__()

        if len(encoder_channels) != 4:
            raise ValueError(
                "encoder_channels must contain exactly 4 values "
                "for c1, c2, c3, c4."
            )

        c1_channels, c2_channels, c3_channels, c4_channels = encoder_channels

        self.c4_projection = nn.Conv2d(
            c4_channels,
            decoder_channels,
            kernel_size=1,
        )
        self.c3_projection = nn.Conv2d(
            c3_channels,
            decoder_channels,
            kernel_size=1,
        )
        self.c2_projection = nn.Conv2d(
            c2_channels,
            decoder_channels,
            kernel_size=1,
        )
        self.c1_projection = nn.Conv2d(
            c1_channels,
            decoder_channels,
            kernel_size=1,
        )

        fusion_channels = decoder_channels * 2

        self.c43_fusion = DecoderBlock(
            in_channels=fusion_channels,
            out_channels=decoder_channels,
            num_groups=num_groups,
        )
        self.d3c2_fusion = DecoderBlock(
            in_channels=fusion_channels,
            out_channels=decoder_channels,
            num_groups=num_groups,
        )
        self.d2c1_fusion = DecoderBlock(
            in_channels=fusion_channels,
            out_channels=decoder_channels,
            num_groups=num_groups,
        )

        self.classifier = nn.Conv2d(
            decoder_channels,
            num_classes,
            kernel_size=1,
        )

    def forward(
        self,
        features,
        output_size,
    ):
        required_keys = ("c1", "c2", "c3", "c4")
        missing_keys = [
            key for key in required_keys
            if key not in features
        ]

        if missing_keys:
            raise KeyError(
                f"Missing encoder feature keys: {missing_keys}"
            )

        c1 = features["c1"]
        c2 = features["c2"]
        c3 = features["c3"]
        c4 = features["c4"]

        # C4 + C3 → D3
        c4 = self.c4_projection(c4)
        c3 = self.c3_projection(c3)

        c4 = F.interpolate(
            c4,
            size=c3.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        d3 = torch.cat([c4, c3], dim=1)
        d3 = self.c43_fusion(d3)

        # D3 + C2 → D2
        c2 = self.c2_projection(c2)

        d3 = F.interpolate(
            d3,
            size=c2.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        d2 = torch.cat([d3, c2], dim=1)
        d2 = self.d3c2_fusion(d2)

        # D2 + C1 → D1
        c1 = self.c1_projection(c1)

        d2 = F.interpolate(
            d2,
            size=c1.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        d1 = torch.cat([d2, c1], dim=1)
        d1 = self.d2c1_fusion(d1)

        # Final logits
        logits = self.classifier(d1)

        logits = F.interpolate(
            logits,
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )

        return logits

## 3-1. Final Head Dummy Forward Test

검증 목표:

- 최종 shape `[B, 3, 448, 768]`
- NaN 없음
- Inf 없음

이 검증은 **Head 구조 자체가 독립적으로 정상 동작하는지** 확인한다.

In [9]:
# ============================================================
# Final SegmentationHead Dummy Forward Test
# ============================================================

dummy_features = {
    "c1": torch.randn(BATCH_SIZE, 96, 112, 192),
    "c2": torch.randn(BATCH_SIZE, 192, 56, 96),
    "c3": torch.randn(BATCH_SIZE, 384, 28, 48),
    "c4": torch.randn(BATCH_SIZE, 768, 14, 24),
}

seg_head = SegmentationHead(
    encoder_channels=(96, 192, 384, 768),
    decoder_channels=256,
    num_classes=3,
    num_groups=32,
)

logits = seg_head(
    dummy_features,
    output_size=(448, 768),
)

print("Segmentation Head output:", logits.shape)
print("NaN:", torch.isnan(logits).any().item())
print("Inf:", torch.isinf(logits).any().item())

assert tuple(logits.shape) == (BATCH_SIZE, 3, 448, 768)
assert torch.isfinite(logits).all().item()

print("Segmentation Head notebook regression: PASS")

Segmentation Head output: torch.Size([2, 3, 448, 768])
NaN: False
Inf: False
Segmentation Head notebook regression: PASS


# 4. `src/segmentation/head.py` 회귀 검증

Notebook 안의 클래스가 아니라 실제 프로젝트 파일을 import해서 다시 검증한다.

이 검증을 하는 이유:

```text
Notebook에서는 성공
 ↓
head.py로 옮기는 과정에서 실수할 수도 있음
 ↓
실제 src 파일을 다시 import해서 검증
```

즉 실제 프로젝트 모듈이 Notebook에서 검증한 구조와 동일하게 동작하는지 확인하는 체크포인트다.

In [10]:
from pathlib import Path
import sys
import json


def find_project_root(start_path=None):
    current = Path(
        start_path if start_path is not None else Path.cwd()
    ).resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "src").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    return None


PROJECT_ROOT = find_project_root()

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\user\Desktop\Accident_AI


In [11]:
# ============================================================
# src/segmentation/head.py Import + Regression Test
# ============================================================

if PROJECT_ROOT is None:
    print("SKIP: 프로젝트 루트를 찾지 못해 src Head regression을 실행하지 않습니다.")
else:
    project_root_str = str(PROJECT_ROOT)

    if project_root_str not in sys.path:
        sys.path.insert(0, project_root_str)

    from src.segmentation.head import (
        SegmentationHead as SegmentationHeadFromSrc,
    )

    dummy_features_src = {
        "c1": torch.randn(BATCH_SIZE, 96, 112, 192),
        "c2": torch.randn(BATCH_SIZE, 192, 56, 96),
        "c3": torch.randn(BATCH_SIZE, 384, 28, 48),
        "c4": torch.randn(BATCH_SIZE, 768, 14, 24),
    }

    seg_head_src = SegmentationHeadFromSrc(
        encoder_channels=(96, 192, 384, 768),
        decoder_channels=256,
        num_classes=3,
        num_groups=32,
    )

    seg_head_src.eval()

    with torch.no_grad():
        logits_src = seg_head_src(
            dummy_features_src,
            output_size=(448, 768),
        )

    shape_ok = tuple(logits_src.shape) == (BATCH_SIZE, 3, 448, 768)
    finite_ok = torch.isfinite(logits_src).all().item()

    print("Output shape:", logits_src.shape)
    print("Shape PASS:", shape_ok)
    print("Finite PASS:", finite_ok)

    assert shape_ok
    assert finite_ok

    print("Segmentation Head src regression: PASS")

Output shape: torch.Size([2, 3, 448, 768])
Shape PASS: True
Finite PASS: True
Segmentation Head src regression: PASS


# 5. Segmentation Loss Design

## Loss란?

Loss는 모델 prediction과 GT가 얼마나 다른지 하나의 숫자로 계산한다.

```text
Seg Head Prediction
[B, 3, H, W]

       +

Segmentation GT
[B, 3, H, W]

       ↓

Loss
```

학습에서는 이 Loss를 줄이는 방향으로 gradient를 계산하고 모델 parameter를 업데이트한다.

## 5-1. 왜 BCE + Dice를 사용하는가?

현재 segmentation은 3개의 **독립 binary channel**이다.

그래서 `softmax`가 아니라 각 채널을 개별적으로 판단해야 한다.

### BCEWithLogitsLoss

```text
raw logits + 0/1 GT
        ↓
BCEWithLogitsLoss
```

- 각 픽셀을 맞췄는지 감독
- background / false positive까지 모두 감독
- 내부적으로 sigmoid와 BCE를 수치적으로 안정적인 방식으로 처리

### Soft Dice Loss

예측 영역과 GT 영역이 얼마나 잘 겹치는지 본다.

```text
Dice Score
=
2 × overlap
──────────────
prediction + GT
```

Dice Loss는:

```text
1 - Dice Score
```

로 만든다.

우리 데이터는 lane / stop_line처럼 얇고 픽셀 수가 적은 구조가 있기 때문에
픽셀 단위 BCE와 영역 겹침 Dice를 함께 쓰는 baseline이 자연스럽다.

### 현재 Baseline

```text
Total Segmentation Loss
=
1.0 × BCE
+
1.0 × Dice
```

1:1 weight는 첫 baseline일 뿐이며 학습 결과에 따라 조정할 수 있다.

## 5-2. 최종 Dice 처리 방식

최종 Soft Dice는 **sample × channel별**로 계산한다.

Tensor:

```text
[B, C, H, W]
```

에서 H/W만 합산해서:

```text
[B, C]
```

Dice를 만든다.

또한 **GT가 실제로 존재하는 sample/channel만 Dice 평균에 포함**한다.

예:

```text
Image 1
lane       있음 → Dice 계산
stop_line  없음 → Dice 제외
crosswalk  있음 → Dice 계산
```

그렇다고 빈 GT에서 false positive를 허용하는 것은 아니다.

- 빈 GT channel의 Dice는 평균에서 제외
- 하지만 BCE는 모든 픽셀을 계속 감독
- 따라서 없는 stop_line을 강하게 예측하면 BCE Loss가 커진다

In [12]:
# ============================================================
# Final Soft Dice Loss
# - sample × channel
# - non-empty GT channel만 Dice 평균에 포함
# ============================================================

def soft_dice_loss(
    logits,
    targets,
    smooth=1.0,
):
    if logits.shape != targets.shape:
        raise ValueError(
            f"logits shape {logits.shape} and "
            f"targets shape {targets.shape} must match."
        )

    if smooth <= 0:
        raise ValueError(
            f"smooth must be greater than 0, got {smooth}."
        )

    probabilities = torch.sigmoid(logits)

    # H, W만 합산 → [B, C]
    dims = (2, 3)

    intersection = (
        probabilities * targets
    ).sum(dim=dims)

    prediction_sum = probabilities.sum(dim=dims)
    target_sum = targets.sum(dim=dims)

    dice_score = (
        2.0 * intersection + smooth
    ) / (
        prediction_sum
        + target_sum
        + smooth
    )

    # GT가 실제 존재하는 sample/channel만 Dice 평균에 포함
    valid_mask = target_sum > 0

    if valid_mask.any():
        valid_dice_score = dice_score[valid_mask]
        dice_loss = (1.0 - valid_dice_score).mean()
    else:
        # 모든 GT channel이 비어 있으면 Dice는 0.
        # BCE가 false positive를 포함한 supervision을 담당한다.
        # logits와 연결된 0을 만들어 gradient graph를 유지한다.
        dice_loss = logits.sum() * 0.0

    return dice_loss

## 5-3. Good / Bad Prediction Test

Loss가 상식적인 방향으로 움직여야 한다.

```text
정답에 거의 맞는 prediction → Loss 작음
정답과 거의 반대 prediction → Loss 큼
```

In [13]:
# ============================================================
# Good / Bad Soft Dice + BCE Test
# ============================================================

dummy_target = torch.tensor(
    [
        [
            [[1.0, 0.0], [0.0, 0.0]],  # traffic_lane
            [[0.0, 1.0], [0.0, 0.0]],  # stop_line
            [[0.0, 0.0], [1.0, 0.0]],  # crosswalk
        ]
    ],
    dtype=torch.float32,
)

good_logits = torch.where(
    dummy_target == 1.0,
    torch.tensor(10.0),
    torch.tensor(-10.0),
)

bad_logits = -good_logits

good_dice = soft_dice_loss(good_logits, dummy_target)
bad_dice = soft_dice_loss(bad_logits, dummy_target)

bce = nn.BCEWithLogitsLoss()
good_bce = bce(good_logits, dummy_target)
bad_bce = bce(bad_logits, dummy_target)

print("Good Dice:", good_dice.item())
print("Bad Dice :", bad_dice.item())
print("Good BCE :", good_bce.item())
print("Bad BCE  :", bad_bce.item())
print("Dice Good < Bad:", good_dice.item() < bad_dice.item())
print("BCE  Good < Bad:", good_bce.item() < bad_bce.item())

assert good_dice.item() < bad_dice.item()
assert good_bce.item() < bad_bce.item()

Good Dice: 6.0558319091796875e-05
Bad Dice : 0.7999781966209412
Good BCE : 4.568200165522285e-05
Bad BCE  : 10.000045776367188
Dice Good < Bad: True
BCE  Good < Bad: True


## 5-4. Empty GT 처리 검증

최종 Dice 설계에서는 **GT가 완전히 빈 channel은 Dice 평균에서 제외**한다.

따라서 GT가 전부 0이라면:

```text
Dice Loss = 0
```

이 되고, good/bad prediction의 차이는 **BCE가 담당**한다.

이 부분은 예전 실험 버전과 다르므로 중요하다.

```text
Empty GT
 ├─ Dice: 둘 다 0으로 제외
 └─ BCE : false positive를 강하게 예측하면 Loss 증가
```

In [14]:
# ============================================================
# Empty GT Test
# - Dice는 제외되어 0
# - false positive 차이는 BCE에서 감독
# ============================================================

empty_target = torch.zeros(
    1, 1, 2, 2,
    dtype=torch.float32,
)

empty_good_logits = torch.full(
    (1, 1, 2, 2),
    -10.0,
)

empty_bad_logits = torch.full(
    (1, 1, 2, 2),
    10.0,
)

empty_good_dice = soft_dice_loss(
    empty_good_logits,
    empty_target,
)

empty_bad_dice = soft_dice_loss(
    empty_bad_logits,
    empty_target,
)

empty_good_bce = bce(
    empty_good_logits,
    empty_target,
)

empty_bad_bce = bce(
    empty_bad_logits,
    empty_target,
)

print("Empty GT Good Dice:", empty_good_dice.item())
print("Empty GT Bad Dice :", empty_bad_dice.item())
print("Empty GT Good BCE :", empty_good_bce.item())
print("Empty GT Bad BCE  :", empty_bad_bce.item())

print("Dice both zero:", (
    empty_good_dice.item() == 0.0
    and empty_bad_dice.item() == 0.0
))
print("BCE Good < Bad:", empty_good_bce.item() < empty_bad_bce.item())

assert empty_good_dice.item() == 0.0
assert empty_bad_dice.item() == 0.0
assert empty_good_bce.item() < empty_bad_bce.item()

Empty GT Good Dice: -0.0
Empty GT Bad Dice : 0.0
Empty GT Good BCE : 4.57763671875e-05
Empty GT Bad BCE  : 10.000045776367188
Dice both zero: True
BCE Good < Bad: True


## 5-5. Mixed sample × channel Test

한 batch 안에서 어떤 class는 존재하고 어떤 class는 비어 있는 실제 상황을 흉내낸다.

```text
Image 1: lane O / stop X / crosswalk O
Image 2: lane O / stop O / crosswalk X
```

Dice는 존재하는 네 개의 sample/channel만 평균에 포함한다.

In [15]:
# ============================================================
# Mixed Sample × Channel Dice Test
# ============================================================

mixed_target = torch.tensor(
    [
        [
            [[1.0, 0.0], [0.0, 0.0]],  # image1 lane O
            [[0.0, 0.0], [0.0, 0.0]],  # image1 stop X
            [[0.0, 0.0], [1.0, 0.0]],  # image1 crosswalk O
        ],
        [
            [[0.0, 1.0], [0.0, 0.0]],  # image2 lane O
            [[0.0, 0.0], [0.0, 1.0]],  # image2 stop O
            [[0.0, 0.0], [0.0, 0.0]],  # image2 crosswalk X
        ],
    ],
    dtype=torch.float32,
)

mixed_good_logits = torch.where(
    mixed_target == 1.0,
    torch.tensor(10.0),
    torch.tensor(-10.0),
)

mixed_bad_logits = -mixed_good_logits

mixed_good_loss = soft_dice_loss(
    mixed_good_logits,
    mixed_target,
)

mixed_bad_loss = soft_dice_loss(
    mixed_bad_logits,
    mixed_target,
)

print("Mixed Good Dice Loss:", mixed_good_loss.item())
print("Mixed Bad Dice Loss :", mixed_bad_loss.item())
print("Good < Bad:", mixed_good_loss.item() < mixed_bad_loss.item())

assert mixed_good_loss.item() < mixed_bad_loss.item()

Mixed Good Dice Loss: 6.054341793060303e-05
Mixed Bad Dice Loss : 0.7999781966209412
Good < Bad: True


## 5-6. Final SegmentationLoss

`SegmentationLoss`는 BCE와 Dice를 함께 계산하고,
학습/로그에서 각각 확인할 수 있도록 dictionary를 반환한다.

```python
{
    "loss": total_loss,
    "bce_loss": bce_loss,
    "dice_loss": dice_loss,
}
```

실제 역전파에는 `losses["loss"]`를 사용한다.

In [16]:
# ============================================================
# Segmentation Loss
# BCEWithLogitsLoss + Soft Dice Loss
# ============================================================

class SegmentationLoss(nn.Module):

    def __init__(
        self,
        bce_weight=1.0,
        dice_weight=1.0,
        smooth=1.0,
    ):
        super().__init__()

        if bce_weight < 0:
            raise ValueError(
                f"bce_weight must be >= 0, got {bce_weight}."
            )

        if dice_weight < 0:
            raise ValueError(
                f"dice_weight must be >= 0, got {dice_weight}."
            )

        if smooth <= 0:
            raise ValueError(
                f"smooth must be greater than 0, got {smooth}."
            )

        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth

        self.bce = nn.BCEWithLogitsLoss()

    def forward(
        self,
        logits,
        targets,
    ):
        if logits.shape != targets.shape:
            raise ValueError(
                f"logits shape {logits.shape} and "
                f"targets shape {targets.shape} must match."
            )

        bce_loss = self.bce(
            logits,
            targets,
        )

        dice_loss = soft_dice_loss(
            logits,
            targets,
            smooth=self.smooth,
        )

        total_loss = (
            self.bce_weight * bce_loss
            + self.dice_weight * dice_loss
        )

        return {
            "loss": total_loss,
            "bce_loss": bce_loss,
            "dice_loss": dice_loss,
        }

## 5-7. Full-size Forward + Backward Test

실제 Head contract와 같은:

```text
[B, 3, 448, 768]
```

크기에서 Loss 계산과 `backward()`까지 확인한다.

검증 목표:

- Loss가 finite
- gradient가 존재
- gradient가 finite
- gradient가 0이 아님

이 네 가지가 통과해야 실제 학습에서 Loss 신호가 모델 쪽으로 전달될 수 있다.

In [17]:
# ============================================================
# Full-size SegmentationLoss Forward + Backward Test
# ============================================================

NUM_CLASSES = 3
HEIGHT = 448
WIDTH = 768

seg_loss_fn = SegmentationLoss(
    bce_weight=1.0,
    dice_weight=1.0,
    smooth=1.0,
)

dummy_logits_grad = torch.randn(
    BATCH_SIZE,
    NUM_CLASSES,
    HEIGHT,
    WIDTH,
    requires_grad=True,
)

dummy_target_grad = torch.randint(
    low=0,
    high=2,
    size=(
        BATCH_SIZE,
        NUM_CLASSES,
        HEIGHT,
        WIDTH,
    ),
).float()

losses = seg_loss_fn(
    dummy_logits_grad,
    dummy_target_grad,
)

total_loss = losses["loss"]
total_loss.backward()

grad_exists = dummy_logits_grad.grad is not None
grad_finite = (
    grad_exists
    and torch.isfinite(dummy_logits_grad.grad).all().item()
)
grad_nonzero = (
    grad_exists
    and dummy_logits_grad.grad.abs().sum().item() > 0.0
)

print("Logits shape:", dummy_logits_grad.shape)
print("Target shape:", dummy_target_grad.shape)
print("Total Loss:", losses["loss"].item())
print("BCE Loss:", losses["bce_loss"].item())
print("Dice Loss:", losses["dice_loss"].item())
print("Loss finite:", torch.isfinite(total_loss).item())
print("Gradient exists:", grad_exists)
print("Gradient finite:", grad_finite)
print("Gradient non-zero:", grad_nonzero)

assert torch.isfinite(total_loss).item()
assert grad_exists
assert grad_finite
assert grad_nonzero

print("Segmentation Loss notebook regression: PASS")

Logits shape: torch.Size([2, 3, 448, 768])
Target shape: torch.Size([2, 3, 448, 768])
Total Loss: 1.3060246706008911
BCE Loss: 0.806148111820221
Dice Loss: 0.4998765289783478
Loss finite: True
Gradient exists: True
Gradient finite: True
Gradient non-zero: True
Segmentation Loss notebook regression: PASS


# 6. `src/segmentation/loss.py` 회귀 검증

실제 프로젝트의 `loss.py`를 import해서
동일한 full-size Forward + Backward 테스트를 수행한다.

이 테스트가 PASS하면:

```text
Notebook Loss 실험
 ↓
loss.py 모듈화
 ↓
실제 src import
 ↓
Forward + Backward
 ↓
PASS
```

까지 확인한 것이다.

In [18]:
# ============================================================
# src/segmentation/loss.py Import + Regression Test
# ============================================================

if PROJECT_ROOT is None:
    print("SKIP: 프로젝트 루트를 찾지 못해 src Loss regression을 실행하지 않습니다.")
else:
    from src.segmentation.loss import (
        SegmentationLoss as SegmentationLossFromSrc,
    )

    logits_src_loss = torch.randn(
        BATCH_SIZE,
        NUM_CLASSES,
        HEIGHT,
        WIDTH,
        requires_grad=True,
    )

    targets_src_loss = torch.randint(
        low=0,
        high=2,
        size=(
            BATCH_SIZE,
            NUM_CLASSES,
            HEIGHT,
            WIDTH,
        ),
    ).float()

    seg_loss_src = SegmentationLossFromSrc(
        bce_weight=1.0,
        dice_weight=1.0,
        smooth=1.0,
    )

    losses_src = seg_loss_src(
        logits_src_loss,
        targets_src_loss,
    )

    total_loss_src = losses_src["loss"]
    total_loss_src.backward()

    shape_ok = logits_src_loss.shape == targets_src_loss.shape
    loss_finite = torch.isfinite(total_loss_src).item()
    grad_exists = logits_src_loss.grad is not None
    grad_finite = (
        grad_exists
        and torch.isfinite(logits_src_loss.grad).all().item()
    )
    grad_nonzero = (
        grad_exists
        and logits_src_loss.grad.abs().sum().item() > 0.0
    )

    print("Logits shape:", logits_src_loss.shape)
    print("Target shape:", targets_src_loss.shape)
    print("Total Loss:", losses_src["loss"].item())
    print("BCE Loss:", losses_src["bce_loss"].item())
    print("Dice Loss:", losses_src["dice_loss"].item())
    print()
    print("Shape PASS:", shape_ok)
    print("Loss finite PASS:", loss_finite)
    print("Gradient exists PASS:", grad_exists)
    print("Gradient finite PASS:", grad_finite)
    print("Gradient non-zero PASS:", grad_nonzero)

    assert shape_ok
    assert loss_finite
    assert grad_exists
    assert grad_finite
    assert grad_nonzero

    print("Segmentation Loss src regression: PASS")

Logits shape: torch.Size([2, 3, 448, 768])
Target shape: torch.Size([2, 3, 448, 768])
Total Loss: 1.3056405782699585
BCE Loss: 0.8058282732963562
Dice Loss: 0.4998123347759247

Shape PASS: True
Loss finite PASS: True
Gradient exists PASS: True
Gradient finite PASS: True
Gradient non-zero PASS: True
Segmentation Loss src regression: PASS


# 7. Dataset / Data Pipeline 설계 — 다음 단계

Head와 Loss baseline은 검증 완료되었다.

이제 Rasterization으로 이미 검증한 실제 데이터를 학습 입력으로 연결한다.

## 현재 목표

한 sample이 최종적으로 대략 다음 형태가 되도록 준비한다.

```python
{
    "image": ...,
    "seg_target": ...,
    "file_name": ...,
}
```

### `image`
- RGB
- `torch.float32`
- `[3, H, W]`
- Shared Encoder 입력용

### `seg_target`
- `torch.float32`
- `[3, H, W]`
- 값: `0.0 / 1.0`
- channel 0: traffic_lane
- channel 1: stop_line
- channel 2: crosswalk

### `file_name`
- QC / debugging / prediction 추적용

## 7-1. Dataset 기본 원칙

### Annotation source
`data/annotations/modelA_segmentation_annotations_v3.jsonl`

### GT 생성
`src/segmentation/rasterizer.py`를 사용해 runtime에 생성한다.

사전에 8,788장의 mask PNG를 전부 저장하지 않는다.

### Transform
RGB와 Segmentation GT는 동일한 `TransformConfig / TransformInfo`를 사용한다.

현재 baseline spatial transform:

```text
1920×1080
→ resize 768×432
→ padding
→ 768×448
```

Dataset 내부에 `768×448`을 직접 hardcoding하지 않는다.

### Normalization
ConvNeXt pretrained normalization은 RGB에만 적용한다.
GT mask에는 적용하지 않는다.

### Annotation Loading
v3 JSONL을 `__getitem__()`마다 다시 읽지 않는다.

초기화 단계에서 한 번 parsing하고,
`file_name` 기준으로 grouping해서 메모리에서 조회한다.

## 7-2. Augmentation 정책

첫 baseline에서는 추가 augmentation을 사용하지 않는다.

사용:
- resize
- padding

사용하지 않음:
- random crop
- horizontal flip
- rotation
- affine
- perspective

이유는 Segmentation과 Vector가 같은 RGB/좌표계를 공유하기 때문이다.

geometry를 바꾸는 augmentation은
향후 RGB / Seg GT / Vector point가 모두 동일하게 변환되는
공통 augmentation pipeline이 마련된 경우에만 검토한다.

## 7-3. Train / Validation Split 정책

Dataset 내부에서 임의로 split하지 않는다.

외부에서 결정된 `file_name` 목록을 Dataset에 전달하는 방향을 유지한다.

가능하면:

1. video / scene / sequence 단위 Group Split
2. 그룹 정보가 없을 경우 deterministic image-level split

을 사용한다.

같은 장면의 매우 유사한 연속 프레임이 Train/Validation 양쪽에 들어가면
Validation 성능이 과대평가될 수 있으므로 주의한다.

In [19]:
# ============================================================
# v3 JSONL Schema Reference
# - Dataset 단계에서 사용할 실제 schema 확인
# ============================================================

if PROJECT_ROOT is None:
    print("SKIP: 프로젝트 루트를 찾지 못했습니다.")
else:
    annotation_path = (
        PROJECT_ROOT
        / "data"
        / "annotations"
        / "modelA_segmentation_annotations_v3.jsonl"
    )

    print("ANNOTATION_PATH:", annotation_path)
    print("annotation exists:", annotation_path.exists())

    if annotation_path.exists():
        with annotation_path.open(
            "r",
            encoding="utf-8",
        ) as f:
            first_line = f.readline()

        first_annotation = json.loads(first_line)

        print()
        print("Top-level keys:")
        print(list(first_annotation.keys()))

        print()
        print("First annotation:")
        print(
            json.dumps(
                first_annotation,
                ensure_ascii=False,
                indent=2,
            )
        )

ANNOTATION_PATH: C:\Users\user\Desktop\Accident_AI\data\annotations\modelA_segmentation_annotations_v3.jsonl
annotation exists: True

Top-level keys:
['file_name', 'image_size', 'annotation_id', 'class', 'lane_type', 'category', 'data']

First annotation:
{
  "file_name": "13883084.jpg",
  "image_size": [
    1080,
    1920
  ],
  "annotation_id": 0,
  "class": "traffic_lane",
  "lane_type": "solid",
  "category": "polyline",
  "data": [
    {
      "x": 207,
      "y": 724
    },
    {
      "x": 206,
      "y": 723
    },
    {
      "x": 480,
      "y": 663
    },
    {
      "x": 832,
      "y": 584
    }
  ]
}


In [20]:
# ============================================================
# Single Image Annotation Grouping Test
# ============================================================

from collections import Counter


# 첫 번째 annotation이 속한 이미지 이름
sample_file_name = first_annotation["file_name"]


# ------------------------------------------------------------
# 같은 file_name의 annotation을 전부 수집
# ------------------------------------------------------------

sample_annotations = []

with annotation_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        annotation = json.loads(line)

        if annotation["file_name"] == sample_file_name:

            sample_annotations.append(
                annotation
            )


# ------------------------------------------------------------
# class / category 개수 확인
# ------------------------------------------------------------

class_counts = Counter(
    annotation["class"]
    for annotation in sample_annotations
)

category_counts = Counter(
    annotation["category"]
    for annotation in sample_annotations
)


# ------------------------------------------------------------
# 결과 출력
# ------------------------------------------------------------

print(
    "Sample file:",
    sample_file_name
)

print(
    "Number of annotations:",
    len(sample_annotations)
)

print()

print(
    "Class counts:",
    dict(class_counts)
)

print(
    "Category counts:",
    dict(category_counts)
)

print()


# annotation별 간단 요약
for annotation in sample_annotations:

    print(
        "annotation_id:",
        annotation["annotation_id"],
        "| class:",
        annotation["class"],
        "| category:",
        annotation["category"],
        "| points:",
        len(annotation["data"]),
    )

Sample file: 13883084.jpg
Number of annotations: 8

Class counts: {'traffic_lane': 8}
Category counts: {'polyline': 8}

annotation_id: 0 | class: traffic_lane | category: polyline | points: 4
annotation_id: 1 | class: traffic_lane | category: polyline | points: 2
annotation_id: 2 | class: traffic_lane | category: polyline | points: 2
annotation_id: 3 | class: traffic_lane | category: polyline | points: 4
annotation_id: 4 | class: traffic_lane | category: polyline | points: 4
annotation_id: 5 | class: traffic_lane | category: polyline | points: 3
annotation_id: 6 | class: traffic_lane | category: polyline | points: 3
annotation_id: 7 | class: traffic_lane | category: polyline | points: 3


In [21]:
# ============================================================
# Representative Single Image Annotation Grouping Test
# - lane / stop_line / crosswalk가 모두 있는 샘플
# ============================================================

representative_file_name = "13886867.jpg"

representative_annotations = []

with annotation_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        annotation = json.loads(line)

        if annotation["file_name"] == representative_file_name:

            representative_annotations.append(
                annotation
            )


# ------------------------------------------------------------
# class / category 개수 확인
# ------------------------------------------------------------

representative_class_counts = Counter(
    annotation["class"]
    for annotation in representative_annotations
)

representative_category_counts = Counter(
    annotation["category"]
    for annotation in representative_annotations
)


# ------------------------------------------------------------
# 결과 출력
# ------------------------------------------------------------

print(
    "Sample file:",
    representative_file_name
)

print(
    "Number of annotations:",
    len(representative_annotations)
)

print()

print(
    "Class counts:",
    dict(representative_class_counts)
)

print(
    "Category counts:",
    dict(representative_category_counts)
)

print()


for annotation in representative_annotations:

    print(
        "annotation_id:",
        annotation["annotation_id"],
        "| class:",
        annotation["class"],
        "| category:",
        annotation["category"],
        "| points:",
        len(annotation["data"]),
    )

Sample file: 13886867.jpg
Number of annotations: 4

Class counts: {'traffic_lane': 1, 'stop_line': 1, 'crosswalk': 2}
Category counts: {'polyline': 2, 'polygon': 2}

annotation_id: 0 | class: traffic_lane | category: polyline | points: 2
annotation_id: 1 | class: stop_line | category: polyline | points: 2
annotation_id: 2 | class: crosswalk | category: polygon | points: 5
annotation_id: 3 | class: crosswalk | category: polygon | points: 8


In [22]:
# ============================================================
# Rasterizer API 확인
# - 기존 rasterizer.py의 실제 함수/클래스 이름과 인자를 확인
# ============================================================

import inspect
import src.segmentation.rasterizer as rasterizer_module


print(
    "Rasterizer module:",
    rasterizer_module.__file__,
)

print()
print("Public API")
print("=" * 60)


for name in dir(rasterizer_module):

    if name.startswith("_"):
        continue

    obj = getattr(
        rasterizer_module,
        name,
    )

    # rasterizer.py 안에서 직접 정의된 함수/클래스만 표시
    if getattr(
        obj,
        "__module__",
        None,
    ) != rasterizer_module.__name__:
        continue

    if (
        inspect.isfunction(obj)
        or inspect.isclass(obj)
    ):

        try:
            signature = inspect.signature(
                obj
            )

        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        print(
            f"{name}{signature}"
        )

Rasterizer module: C:\Users\user\Desktop\Accident_AI\src\segmentation\rasterizer.py

Public API
RasterConfig(lane_thickness: int = 3, stop_thickness: int = 3) -> None
make_raster_stats() -> dict[str, int]
prepare_source_points(data: list[dict], original_width: int, original_height: int, stats: dict[str, int]) -> numpy.ndarray | None
rasterize_segmentation_targets(annotations: list[dict], original_width: int, original_height: int, transform_info: src.preprocessing.transforms.TransformInfo, raster_config: src.segmentation.rasterizer.RasterConfig = RasterConfig(lane_thickness=3, stop_thickness=3)) -> dict
to_raster_points(source_points: numpy.ndarray, transform_info: src.preprocessing.transforms.TransformInfo) -> numpy.ndarray


In [23]:
# ============================================================
# Transform API 확인
# - TransformInfo가 어떻게 만들어지는지 확인
# ============================================================

import inspect
import src.preprocessing.transforms as transforms_module


print(
    "Transforms module:",
    transforms_module.__file__,
)

print()
print("Public API")
print("=" * 60)


for name in dir(transforms_module):

    if name.startswith("_"):
        continue

    obj = getattr(
        transforms_module,
        name,
    )

    # transforms.py 안에서 직접 정의된 것만 표시
    if getattr(
        obj,
        "__module__",
        None,
    ) != transforms_module.__name__:
        continue

    if (
        inspect.isfunction(obj)
        or inspect.isclass(obj)
    ):

        try:
            signature = inspect.signature(obj)

        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        print(
            f"{name}{signature}"
        )

Transforms module: C:\Users\user\Desktop\Accident_AI\src\preprocessing\transforms.py

Public API
TransformConfig(resize_scale: float = 0.4, model_stride: int = 32, pad_value: tuple[int, int, int] = (0, 0, 0)) -> None
TransformInfo(scale_x: float, scale_y: float, original_width: int, original_height: int, resized_width: int, resized_height: int, padded_width: int, padded_height: int, pad_left: int, pad_right: int, pad_top: int, pad_bottom: int) -> None
ceil_to_multiple(value: int, multiple: int) -> int
compute_transform_info(original_width: int, original_height: int, config: src.preprocessing.transforms.TransformConfig = TransformConfig(resize_scale=0.4, model_stride=32, pad_value=(0, 0, 0))) -> src.preprocessing.transforms.TransformInfo
transform_image(image_bgr: numpy.ndarray, transform_info: src.preprocessing.transforms.TransformInfo, config: src.preprocessing.transforms.TransformConfig = TransformConfig(resize_scale=0.4, model_stride=32, pad_value=(0, 0, 0))) -> numpy.ndarray
transf

In [2]:
# ============================================================
# Single Sample TransformInfo Test
# - Project Root 설정 포함
# ============================================================

from pathlib import Path
import sys


# ------------------------------------------------------------
# 1. Project Root 찾기
#
# Accident_AI/
# ├─ src/
# ├─ data/
# └─ notebooks/
#
# 같은 구조를 찾아서 Project Root로 사용
# ------------------------------------------------------------

def find_project_root(start_path=None):

    current = Path(
        start_path if start_path is not None
        else Path.cwd()
    ).resolve()

    for candidate in [
        current,
        *current.parents,
    ]:

        if (
            (candidate / "src").exists()
            and
            (candidate / "data").exists()
        ):
            return candidate

    return None


PROJECT_ROOT = find_project_root()


if PROJECT_ROOT is None:
    raise RuntimeError(
        "PROJECT_ROOT를 찾지 못했습니다."
    )


# ------------------------------------------------------------
# 2. Python이 src를 찾을 수 있도록
#    Accident_AI 폴더를 import path에 추가
# ------------------------------------------------------------

project_root_str = str(PROJECT_ROOT)

if project_root_str not in sys.path:

    sys.path.insert(
        0,
        project_root_str,
    )


print(
    "PROJECT_ROOT:",
    PROJECT_ROOT
)


# ------------------------------------------------------------
# 3. 이제 src import
# ------------------------------------------------------------

from src.preprocessing.transforms import (
    TransformConfig,
    compute_transform_info,
)


# ------------------------------------------------------------
# 4. Model A 공통 Transform 설정
# ------------------------------------------------------------

transform_config = TransformConfig(
    resize_scale=0.4,
    model_stride=32,
    pad_value=(0, 0, 0),
)


# ------------------------------------------------------------
# 5. 원본 이미지 크기
#
# JSONL image_size:
# [1080, 1920]
#
# height = 1080
# width  = 1920
# ------------------------------------------------------------

original_height = 1080
original_width = 1920


# ------------------------------------------------------------
# 6. TransformInfo 계산
# ------------------------------------------------------------

transform_info = compute_transform_info(
    original_width=original_width,
    original_height=original_height,
    config=transform_config,
)


# ------------------------------------------------------------
# 7. 결과 확인
# ------------------------------------------------------------

print()
print(
    "Transform config:"
)
print(
    transform_config
)

print()
print(
    "Transform info:"
)
print(
    transform_info
)

PROJECT_ROOT: C:\Users\user\Desktop\Accident_AI

Transform config:
TransformConfig(resize_scale=0.4, model_stride=32, pad_value=(0, 0, 0))

Transform info:
TransformInfo(scale_x=0.4, scale_y=0.4, original_width=1920, original_height=1080, resized_width=768, resized_height=432, padded_width=768, padded_height=448, pad_left=0, pad_right=0, pad_top=8, pad_bottom=8)


In [3]:
# ============================================================
# TransformInfo 상세 확인
# ============================================================

print("scale_x       :", transform_info.scale_x)
print("scale_y       :", transform_info.scale_y)

print()

print("original size :",
      transform_info.original_width,
      "x",
      transform_info.original_height)

print("resized size  :",
      transform_info.resized_width,
      "x",
      transform_info.resized_height)

print("padded size   :",
      transform_info.padded_width,
      "x",
      transform_info.padded_height)

print()

print("pad_left      :", transform_info.pad_left)
print("pad_right     :", transform_info.pad_right)
print("pad_top       :", transform_info.pad_top)
print("pad_bottom    :", transform_info.pad_bottom)

scale_x       : 0.4
scale_y       : 0.4

original size : 1920 x 1080
resized size  : 768 x 432
padded size   : 768 x 448

pad_left      : 0
pad_right     : 0
pad_top       : 8
pad_bottom    : 8


In [6]:
# ============================================================
# rasterize_segmentation_targets 정확한 API 확인
# ============================================================

import inspect

from src.segmentation.rasterizer import (
    rasterize_segmentation_targets,
)


signature = inspect.signature(
    rasterize_segmentation_targets
)

print(
    "Function:",
    rasterize_segmentation_targets.__name__
)

print()
print("Parameters")
print("=" * 60)

for name, parameter in signature.parameters.items():

    print(
        f"{name}"
    )

    print(
        "  annotation:",
        parameter.annotation
    )

    print(
        "  default:",
        parameter.default
    )

    print()


print("Return annotation")
print("=" * 60)

print(
    signature.return_annotation
)

Function: rasterize_segmentation_targets

Parameters
annotations
  annotation: list[dict]
  default: <class 'inspect._empty'>

original_width
  annotation: <class 'int'>
  default: <class 'inspect._empty'>

original_height
  annotation: <class 'int'>
  default: <class 'inspect._empty'>

transform_info
  annotation: <class 'src.preprocessing.transforms.TransformInfo'>
  default: <class 'inspect._empty'>

raster_config
  annotation: <class 'src.segmentation.rasterizer.RasterConfig'>
  default: RasterConfig(lane_thickness=3, stop_thickness=3)

Return annotation
<class 'dict'>


In [8]:
# ============================================================
# Representative Sample Rasterization
# - 필요한 변수들을 이 셀에서 모두 다시 준비
# ============================================================

from pathlib import Path
import sys
import json
import numpy as np


# ------------------------------------------------------------
# 1. PROJECT_ROOT 찾기
# ------------------------------------------------------------

def find_project_root(start_path=None):

    current = Path(
        start_path if start_path is not None
        else Path.cwd()
    ).resolve()

    for candidate in [current, *current.parents]:

        if (
            (candidate / "src").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    return None


PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None:
    raise RuntimeError(
        "PROJECT_ROOT를 찾지 못했습니다."
    )


project_root_str = str(PROJECT_ROOT)

if project_root_str not in sys.path:
    sys.path.insert(
        0,
        project_root_str,
    )


print(
    "PROJECT_ROOT:",
    PROJECT_ROOT,
)


# ------------------------------------------------------------
# 2. 필요한 실제 프로젝트 코드 import
# ------------------------------------------------------------

from src.preprocessing.transforms import (
    TransformConfig,
    compute_transform_info,
)

from src.segmentation.rasterizer import (
    RasterConfig,
    rasterize_segmentation_targets,
)


# ------------------------------------------------------------
# 3. Annotation 파일
# ------------------------------------------------------------

annotation_path = (
    PROJECT_ROOT
    / "data"
    / "annotations"
    / "modelA_segmentation_annotations_v3.jsonl"
)

if not annotation_path.exists():
    raise FileNotFoundError(
        f"Annotation file not found: {annotation_path}"
    )


# ------------------------------------------------------------
# 4. 대표 이미지의 annotation 4개 다시 수집
# ------------------------------------------------------------

representative_file_name = "13886867.jpg"

representative_annotations = []

with annotation_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        annotation = json.loads(line)

        if (
            annotation["file_name"]
            == representative_file_name
        ):

            representative_annotations.append(
                annotation
            )


print()
print(
    "Sample file:",
    representative_file_name,
)

print(
    "Number of annotations:",
    len(representative_annotations),
)


if len(representative_annotations) == 0:
    raise RuntimeError(
        "대표 이미지 annotation을 찾지 못했습니다."
    )


# ------------------------------------------------------------
# 5. 실제 JSONL의 image_size 사용
#
# image_size = [height, width]
# ------------------------------------------------------------

original_height = (
    representative_annotations[0]["image_size"][0]
)

original_width = (
    representative_annotations[0]["image_size"][1]
)


print(
    "Original size:",
    original_width,
    "x",
    original_height,
)


# ------------------------------------------------------------
# 6. TransformInfo 생성
# ------------------------------------------------------------

transform_config = TransformConfig(
    resize_scale=0.4,
    model_stride=32,
    pad_value=(0, 0, 0),
)

transform_info = compute_transform_info(
    original_width=original_width,
    original_height=original_height,
    config=transform_config,
)


print(
    "Padded size:",
    transform_info.padded_width,
    "x",
    transform_info.padded_height,
)


# ------------------------------------------------------------
# 7. RasterConfig
# ------------------------------------------------------------

raster_config = RasterConfig(
    lane_thickness=3,
    stop_thickness=3,
)


# ------------------------------------------------------------
# 8. 실제 Rasterization
# ------------------------------------------------------------

raster_result = rasterize_segmentation_targets(
    annotations=representative_annotations,
    original_width=original_width,
    original_height=original_height,
    transform_info=transform_info,
    raster_config=raster_config,
)


# ------------------------------------------------------------
# 9. 반환 구조 확인
# ------------------------------------------------------------

print()
print(
    "Result type:",
    type(raster_result),
)

print(
    "Result keys:",
    list(raster_result.keys()),
)

print()
print("=" * 70)


for key, value in raster_result.items():

    print()
    print(
        f"[{key}]"
    )

    print(
        "type:",
        type(value),
    )

    if isinstance(value, np.ndarray):

        print(
            "shape:",
            value.shape,
        )

        print(
            "dtype:",
            value.dtype,
        )

        print(
            "min:",
            value.min(),
        )

        print(
            "max:",
            value.max(),
        )

        print(
            "non-zero pixels:",
            np.count_nonzero(value),
        )

    else:

        print(
            "value:",
            value,
        )

PROJECT_ROOT: C:\Users\user\Desktop\Accident_AI

Sample file: 13886867.jpg
Number of annotations: 4
Original size: 1920 x 1080
Padded size: 768 x 448

Result type: <class 'dict'>
Result keys: ['lane_mask', 'stop_mask', 'crosswalk_mask', 'stats']


[lane_mask]
type: <class 'numpy.ndarray'>
shape: (448, 768)
dtype: uint8
min: 0
max: 255
non-zero pixels: 487

[stop_mask]
type: <class 'numpy.ndarray'>
shape: (448, 768)
dtype: uint8
min: 0
max: 255
non-zero pixels: 158

[crosswalk_mask]
type: <class 'numpy.ndarray'>
shape: (448, 768)
dtype: uint8
min: 0
max: 255
non-zero pixels: 4431

[stats]
type: <class 'dict'>
value: {'traffic_lane_drawn': 1, 'stop_line_drawn': 1, 'crosswalk_drawn': 2, 'empty_data_skipped': 0, 'invalid_polyline_skipped': 0, 'invalid_polygon_skipped': 0, 'source_points_clipped': 0}


In [9]:
# ============================================================
# 3-channel Segmentation GT 만들기
# ============================================================

import numpy as np
import torch


# ------------------------------------------------------------
# 1. Rasterizer 결과에서 각 mask 가져오기
# ------------------------------------------------------------

lane_mask = raster_result["lane_mask"]
stop_mask = raster_result["stop_mask"]
crosswalk_mask = raster_result["crosswalk_mask"]


# ------------------------------------------------------------
# 2. 각 mask 기본 검증
# ------------------------------------------------------------

expected_shape = (
    transform_info.padded_height,
    transform_info.padded_width,
)

print("Expected mask shape:", expected_shape)
print()

print(
    "lane      :",
    lane_mask.shape,
    lane_mask.dtype,
    "pixels:",
    np.count_nonzero(lane_mask),
)

print(
    "stop_line :",
    stop_mask.shape,
    stop_mask.dtype,
    "pixels:",
    np.count_nonzero(stop_mask),
)

print(
    "crosswalk :",
    crosswalk_mask.shape,
    crosswalk_mask.dtype,
    "pixels:",
    np.count_nonzero(crosswalk_mask),
)


assert lane_mask.shape == expected_shape
assert stop_mask.shape == expected_shape
assert crosswalk_mask.shape == expected_shape


# ------------------------------------------------------------
# 3. Channel 순서대로 Stack
#
# channel 0 = traffic_lane
# channel 1 = stop_line
# channel 2 = crosswalk
#
# [H,W] 3개
#    ↓
# [3,H,W]
# ------------------------------------------------------------

seg_target_np = np.stack(
    [
        lane_mask,
        stop_mask,
        crosswalk_mask,
    ],
    axis=0,
)


# ------------------------------------------------------------
# 4. uint8 0/255
#       ↓
#    float32 0/1
# ------------------------------------------------------------

seg_target_np = (
    seg_target_np.astype(np.float32)
    / 255.0
)


# ------------------------------------------------------------
# 5. PyTorch Tensor로 변환
# ------------------------------------------------------------

seg_target = torch.from_numpy(
    seg_target_np
)


# ------------------------------------------------------------
# 6. 최종 확인
# ------------------------------------------------------------

print()
print("=" * 60)

print(
    "seg_target shape:",
    seg_target.shape,
)

print(
    "seg_target dtype:",
    seg_target.dtype,
)

print(
    "seg_target min:",
    seg_target.min().item(),
)

print(
    "seg_target max:",
    seg_target.max().item(),
)

print()

print(
    "channel 0 lane pixels:",
    torch.count_nonzero(
        seg_target[0]
    ).item(),
)

print(
    "channel 1 stop pixels:",
    torch.count_nonzero(
        seg_target[1]
    ).item(),
)

print(
    "channel 2 crosswalk pixels:",
    torch.count_nonzero(
        seg_target[2]
    ).item(),
)


assert tuple(seg_target.shape) == (
    3,
    448,
    768,
)

assert seg_target.dtype == torch.float32

assert (
    seg_target.min().item() >= 0.0
)

assert (
    seg_target.max().item() <= 1.0
)


print()
print(
    "Single Sample Segmentation GT: PASS"
)

Expected mask shape: (448, 768)

lane      : (448, 768) uint8 pixels: 487
stop_line : (448, 768) uint8 pixels: 158
crosswalk : (448, 768) uint8 pixels: 4431

seg_target shape: torch.Size([3, 448, 768])
seg_target dtype: torch.float32
seg_target min: 0.0
seg_target max: 1.0

channel 0 lane pixels: 487
channel 1 stop pixels: 158
channel 2 crosswalk pixels: 4431

Single Sample Segmentation GT: PASS


In [10]:
# ============================================================
# Representative RGB Image Path 확인
# ============================================================

representative_file_name = "13886867.jpg"


# ------------------------------------------------------------
# data 폴더 아래에서 실제 이미지 찾기
# ------------------------------------------------------------

image_matches = list(
    (PROJECT_ROOT / "data").rglob(
        representative_file_name
    )
)


print(
    "Number of image matches:",
    len(image_matches),
)

print()

for image_path in image_matches:

    print(
        "Image path:",
        image_path
    )

Number of image matches: 1

Image path: C:\Users\user\Desktop\Accident_AI\data\raw\images\train_09\13886867.jpg


In [11]:
# ============================================================
# Representative RGB Image → Tensor
# - 실제 이미지 읽기
# - GT와 동일한 spatial Transform 적용
# ============================================================

import cv2
import numpy as np
import torch

from src.preprocessing.transforms import (
    transform_image,
)


# ------------------------------------------------------------
# 1. 실제 이미지 경로
# ------------------------------------------------------------

image_path = image_matches[0]

print(
    "Image path:",
    image_path,
)


# ------------------------------------------------------------
# 2. OpenCV로 원본 이미지 읽기
#
# cv2.imread() 결과는 BGR 순서
# ------------------------------------------------------------

image_bgr = cv2.imread(
    str(image_path),
    cv2.IMREAD_COLOR,
)

if image_bgr is None:
    raise RuntimeError(
        f"이미지를 읽지 못했습니다: {image_path}"
    )


print()
print(
    "Original image shape:",
    image_bgr.shape,
)

print(
    "Original image dtype:",
    image_bgr.dtype,
)


# ------------------------------------------------------------
# 3. 실제 annotation image_size와 일치하는지 확인
#
# OpenCV shape:
# [H, W, C]
# ------------------------------------------------------------

assert image_bgr.shape[:2] == (
    original_height,
    original_width,
)


# ------------------------------------------------------------
# 4. GT와 동일한 TransformInfo 적용
#
# 1920×1080
#   ↓
# 768×432
#   ↓ padding
# 768×448
# ------------------------------------------------------------

image_bgr_transformed = transform_image(
    image_bgr=image_bgr,
    transform_info=transform_info,
    config=transform_config,
)


print()
print(
    "Transformed image shape:",
    image_bgr_transformed.shape,
)

print(
    "Transformed image dtype:",
    image_bgr_transformed.dtype,
)


# ------------------------------------------------------------
# 5. BGR → RGB
# ------------------------------------------------------------

image_rgb = cv2.cvtColor(
    image_bgr_transformed,
    cv2.COLOR_BGR2RGB,
)


# ------------------------------------------------------------
# 6. NumPy [H,W,C]
#       ↓
#    Tensor [C,H,W]
#
# uint8 0~255
#       ↓
# float32 0~1
# ------------------------------------------------------------

image_tensor = torch.from_numpy(
    image_rgb.copy()
).permute(
    2,
    0,
    1,
).float() / 255.0


# ------------------------------------------------------------
# 7. 최종 RGB Tensor 확인
# ------------------------------------------------------------

print()
print("=" * 60)

print(
    "image_tensor shape:",
    image_tensor.shape,
)

print(
    "image_tensor dtype:",
    image_tensor.dtype,
)

print(
    "image_tensor min:",
    image_tensor.min().item(),
)

print(
    "image_tensor max:",
    image_tensor.max().item(),
)


# ------------------------------------------------------------
# 8. GT와 spatial shape가 정확히 같은지 확인
# ------------------------------------------------------------

spatial_match = (
    image_tensor.shape[-2:]
    == seg_target.shape[-2:]
)

print()
print(
    "Image / GT spatial match:",
    spatial_match,
)


assert tuple(image_tensor.shape) == (
    3,
    448,
    768,
)

assert image_tensor.dtype == torch.float32
assert spatial_match


print()
print(
    "Single Sample RGB Transform: PASS"
)

Image path: C:\Users\user\Desktop\Accident_AI\data\raw\images\train_09\13886867.jpg

Original image shape: (1080, 1920, 3)
Original image dtype: uint8

Transformed image shape: (448, 768, 3)
Transformed image dtype: uint8

image_tensor shape: torch.Size([3, 448, 768])
image_tensor dtype: torch.float32
image_tensor min: 0.0
image_tensor max: 1.0

Image / GT spatial match: True

Single Sample RGB Transform: PASS


In [12]:
# ============================================================
# Single Segmentation Sample 만들기
# ============================================================

single_sample = {
    "image": image_tensor,
    "seg_target": seg_target,
    "file_name": representative_file_name,
}


# ------------------------------------------------------------
# Sample 구조 확인
# ------------------------------------------------------------

print(
    "Sample keys:",
    list(single_sample.keys()),
)

print()

print(
    "file_name:",
    single_sample["file_name"],
)

print(
    "image shape:",
    single_sample["image"].shape,
)

print(
    "image dtype:",
    single_sample["image"].dtype,
)

print(
    "seg_target shape:",
    single_sample["seg_target"].shape,
)

print(
    "seg_target dtype:",
    single_sample["seg_target"].dtype,
)


# ------------------------------------------------------------
# 최종 Contract 검증
# ------------------------------------------------------------

image_ok = (
    tuple(single_sample["image"].shape)
    == (3, 448, 768)
)

target_ok = (
    tuple(single_sample["seg_target"].shape)
    == (3, 448, 768)
)

dtype_ok = (
    single_sample["image"].dtype
    == torch.float32
    and
    single_sample["seg_target"].dtype
    == torch.float32
)

spatial_match = (
    single_sample["image"].shape[-2:]
    ==
    single_sample["seg_target"].shape[-2:]
)


print()
print("=" * 60)

print(
    "Image shape PASS:",
    image_ok,
)

print(
    "Target shape PASS:",
    target_ok,
)

print(
    "Dtype PASS:",
    dtype_ok,
)

print(
    "Spatial match PASS:",
    spatial_match,
)


assert image_ok
assert target_ok
assert dtype_ok
assert spatial_match


print()
print(
    "Single Segmentation Sample: PASS"
)

Sample keys: ['image', 'seg_target', 'file_name']

file_name: 13886867.jpg
image shape: torch.Size([3, 448, 768])
image dtype: torch.float32
seg_target shape: torch.Size([3, 448, 768])
seg_target dtype: torch.float32

Image shape PASS: True
Target shape PASS: True
Dtype PASS: True
Spatial match PASS: True

Single Segmentation Sample: PASS


In [13]:
# ============================================================
# Full Dataset Index Test
# - JSONL annotation을 file_name별로 grouping
# - 실제 image path를 file_name별로 indexing
# - 누락 / 중복 여부 확인
# ============================================================

from collections import defaultdict
from pathlib import Path
import json


# ------------------------------------------------------------
# 1. Annotation file
# ------------------------------------------------------------

annotation_path = (
    PROJECT_ROOT
    / "data"
    / "annotations"
    / "modelA_segmentation_annotations_v3.jsonl"
)


# ------------------------------------------------------------
# 2. 모든 annotation을 file_name별로 grouping
#
# 예:
#
# annotation_groups["13886867.jpg"]
#     → annotation 4개
# ------------------------------------------------------------

annotation_groups = defaultdict(list)


with annotation_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line in f:

        annotation = json.loads(line)

        file_name = annotation["file_name"]

        annotation_groups[file_name].append(
            annotation
        )


# 일반 dict로 변환
annotation_groups = dict(
    annotation_groups
)


# ------------------------------------------------------------
# 3. 실제 image 파일 전체 탐색
#
# 파일명 하나에 경로가 여러 개 있을 가능성도
# 검출할 수 있도록 list로 저장
# ------------------------------------------------------------

image_root = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "images"
)


image_candidates = defaultdict(list)


for image_path in image_root.rglob("*"):

    if not image_path.is_file():
        continue

    if image_path.suffix.lower() not in {
        ".jpg",
        ".jpeg",
        ".png",
    }:
        continue

    image_candidates[
        image_path.name
    ].append(
        image_path
    )


# ------------------------------------------------------------
# 4. Annotation에 존재하는 이미지들만 검사
# ------------------------------------------------------------

missing_images = []

duplicate_images = []

image_path_map = {}


for file_name in annotation_groups:

    matches = image_candidates.get(
        file_name,
        [],
    )

    if len(matches) == 0:

        missing_images.append(
            file_name
        )

    elif len(matches) > 1:

        duplicate_images.append(
            (
                file_name,
                matches,
            )
        )

    else:

        image_path_map[
            file_name
        ] = matches[0]


# ------------------------------------------------------------
# 5. 전체 상태 출력
# ------------------------------------------------------------

total_annotations = sum(
    len(annotations)
    for annotations
    in annotation_groups.values()
)


print(
    "Unique annotated images:",
    len(annotation_groups),
)

print(
    "Total annotations:",
    total_annotations,
)

print(
    "Resolved image paths:",
    len(image_path_map),
)

print()

print(
    "Missing images:",
    len(missing_images),
)

print(
    "Duplicate image names:",
    len(duplicate_images),
)


# ------------------------------------------------------------
# 6. 대표 이미지도 제대로 매핑됐는지 확인
# ------------------------------------------------------------

print()
print("=" * 70)

representative_file_name = "13886867.jpg"

print(
    "Representative annotations:",
    len(
        annotation_groups[
            representative_file_name
        ]
    ),
)

print(
    "Representative image path:",
    image_path_map.get(
        representative_file_name
    ),
)


# ------------------------------------------------------------
# 7. 안전성 검증
# ------------------------------------------------------------

assert len(missing_images) == 0
assert len(duplicate_images) == 0

assert (
    len(image_path_map)
    ==
    len(annotation_groups)
)


print()
print(
    "Full Dataset Index: PASS"
)

Unique annotated images: 8788
Total annotations: 52208
Resolved image paths: 8788

Missing images: 0
Duplicate image names: 0

Representative annotations: 4
Representative image path: C:\Users\user\Desktop\Accident_AI\data\raw\images\train_09\13886867.jpg

Full Dataset Index: PASS


In [14]:
# ============================================================
# SegmentationDataset Baseline
#
# 역할:
# file_name 하나를 받았을 때
#
# image      : [3, H, W] float32
# seg_target : [3, H, W] float32
# file_name  : str
#
# 형태의 sample을 반환
# ============================================================

import cv2
import json
import numpy as np
import torch

from collections import defaultdict
from pathlib import Path
from torch.utils.data import Dataset

from src.preprocessing.transforms import (
    TransformConfig,
    compute_transform_info,
    transform_image,
)

from src.segmentation.rasterizer import (
    RasterConfig,
    rasterize_segmentation_targets,
)


class SegmentationDataset(Dataset):

    def __init__(
        self,
        annotation_path,
        image_root,
        file_names=None,
        transform_config=None,
        raster_config=None,
    ):
        super().__init__()

        self.annotation_path = Path(
            annotation_path
        )

        self.image_root = Path(
            image_root
        )


        # ----------------------------------------------------
        # Transform / Raster 설정
        #
        # 외부에서 설정을 주지 않으면
        # 현재 Model A baseline 값을 사용
        # ----------------------------------------------------

        if transform_config is None:

            transform_config = TransformConfig(
                resize_scale=0.4,
                model_stride=32,
                pad_value=(0, 0, 0),
            )

        if raster_config is None:

            raster_config = RasterConfig(
                lane_thickness=3,
                stop_thickness=3,
            )


        self.transform_config = (
            transform_config
        )

        self.raster_config = (
            raster_config
        )


        # ----------------------------------------------------
        # 1. JSONL을 한 번만 읽어서
        #    file_name별 annotation grouping
        #
        # __getitem__ 호출 때마다
        # JSONL 전체를 다시 읽지 않음
        # ----------------------------------------------------

        annotation_groups = defaultdict(
            list
        )


        with self.annotation_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                annotation = json.loads(
                    line
                )

                annotation_groups[
                    annotation["file_name"]
                ].append(
                    annotation
                )


        self.annotation_groups = dict(
            annotation_groups
        )


        # ----------------------------------------------------
        # 2. Dataset에서 사용할 file_name 결정
        #
        # file_names를 외부에서 주면
        # 그 목록만 사용
        #
        # 나중에 Train / Validation Split을
        # Dataset 바깥에서 관리하기 위함
        # ----------------------------------------------------

        if file_names is None:

            self.file_names = sorted(
                self.annotation_groups.keys()
            )

        else:

            self.file_names = list(
                file_names
            )


        # ----------------------------------------------------
        # 3. 요청된 file_name이 annotation에
        #    실제 존재하는지 검증
        # ----------------------------------------------------

        missing_annotations = [

            file_name

            for file_name
            in self.file_names

            if file_name
            not in self.annotation_groups
        ]


        if missing_annotations:

            raise ValueError(
                "Annotation이 없는 file_name이 있습니다. "
                f"예: {missing_annotations[:5]}"
            )


        # ----------------------------------------------------
        # 4. 실제 RGB image path index 생성
        #
        # file_name → 실제 경로
        # ----------------------------------------------------

        requested_names = set(
            self.file_names
        )

        image_candidates = defaultdict(
            list
        )


        for image_path in self.image_root.rglob(
            "*"
        ):

            if not image_path.is_file():
                continue

            if image_path.suffix.lower() not in {
                ".jpg",
                ".jpeg",
                ".png",
            }:
                continue

            if image_path.name not in requested_names:
                continue

            image_candidates[
                image_path.name
            ].append(
                image_path
            )


        self.image_path_map = {}


        for file_name in self.file_names:

            matches = image_candidates.get(
                file_name,
                [],
            )


            if len(matches) == 0:

                raise FileNotFoundError(
                    f"Image를 찾지 못했습니다: "
                    f"{file_name}"
                )


            if len(matches) > 1:

                raise RuntimeError(
                    f"동일한 file_name의 image가 "
                    f"여러 개 있습니다: "
                    f"{file_name}"
                )


            self.image_path_map[
                file_name
            ] = matches[0]


    # ========================================================
    # Dataset 길이
    # ========================================================

    def __len__(self):

        return len(
            self.file_names
        )


    # ========================================================
    # index 하나 → sample 하나
    # ========================================================

    def __getitem__(
        self,
        index,
    ):

        # ----------------------------------------------------
        # 1. file_name / annotation / image path
        # ----------------------------------------------------

        file_name = self.file_names[
            index
        ]

        annotations = (
            self.annotation_groups[
                file_name
            ]
        )

        image_path = (
            self.image_path_map[
                file_name
            ]
        )


        # ----------------------------------------------------
        # 2. 실제 RGB 읽기
        #
        # OpenCV는 BGR
        # ----------------------------------------------------

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )


        if image_bgr is None:

            raise RuntimeError(
                f"Image를 읽지 못했습니다: "
                f"{image_path}"
            )


        actual_height, actual_width = (
            image_bgr.shape[:2]
        )


        # ----------------------------------------------------
        # 3. Annotation의 원본 image_size 확인
        #
        # JSONL:
        # [height, width]
        # ----------------------------------------------------

        annotation_height = (
            annotations[0][
                "image_size"
            ][0]
        )

        annotation_width = (
            annotations[0][
                "image_size"
            ][1]
        )


        if (
            actual_height
            != annotation_height
            or
            actual_width
            != annotation_width
        ):

            raise ValueError(
                f"Image size mismatch: "
                f"{file_name} | "
                f"image="
                f"{actual_width}x{actual_height}, "
                f"annotation="
                f"{annotation_width}x"
                f"{annotation_height}"
            )


        # ----------------------------------------------------
        # 4. 공통 TransformInfo
        #
        # RGB와 GT가 반드시
        # 동일한 TransformInfo 사용
        # ----------------------------------------------------

        transform_info = (
            compute_transform_info(
                original_width=actual_width,
                original_height=actual_height,
                config=self.transform_config,
            )
        )


        # ----------------------------------------------------
        # 5. RGB Transform
        # ----------------------------------------------------

        image_bgr = transform_image(
            image_bgr=image_bgr,
            transform_info=transform_info,
            config=self.transform_config,
        )


        # BGR → RGB
        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )


        # [H,W,C]
        #    ↓
        # [C,H,W]
        #
        # uint8 0~255
        #    ↓
        # float32 0~1
        image_tensor = (
            torch.from_numpy(
                image_rgb.copy()
            )
            .permute(
                2,
                0,
                1,
            )
            .float()
            / 255.0
        )


        # ----------------------------------------------------
        # 6. Segmentation GT Rasterization
        # ----------------------------------------------------

        raster_result = (
            rasterize_segmentation_targets(
                annotations=annotations,
                original_width=actual_width,
                original_height=actual_height,
                transform_info=transform_info,
                raster_config=self.raster_config,
            )
        )


        # ----------------------------------------------------
        # 7. 세 mask를 channel 순서대로 Stack
        #
        # channel 0 = traffic_lane
        # channel 1 = stop_line
        # channel 2 = crosswalk
        # ----------------------------------------------------

        seg_target_np = np.stack(
            [
                raster_result[
                    "lane_mask"
                ],
                raster_result[
                    "stop_mask"
                ],
                raster_result[
                    "crosswalk_mask"
                ],
            ],
            axis=0,
        )


        # uint8 0/255
        #     ↓
        # float32 0/1
        seg_target_np = (
            seg_target_np.astype(
                np.float32
            )
            / 255.0
        )


        seg_target = torch.from_numpy(
            seg_target_np
        )


        # ----------------------------------------------------
        # 8. RGB ↔ GT spatial contract 검사
        # ----------------------------------------------------

        if (
            image_tensor.shape[-2:]
            !=
            seg_target.shape[-2:]
        ):

            raise RuntimeError(
                f"Image / GT spatial mismatch: "
                f"{file_name}"
            )


        # ----------------------------------------------------
        # 9. 한 sample 반환
        # ----------------------------------------------------

        return {
            "image": image_tensor,
            "seg_target": seg_target,
            "file_name": file_name,
        }

In [15]:
# ============================================================
# SegmentationDataset Single Sample Regression Test
# ============================================================

representative_file_name = "13886867.jpg"


# ------------------------------------------------------------
# 1. 대표 이미지 1장만 사용하는 Dataset 생성
# ------------------------------------------------------------

single_dataset = SegmentationDataset(
    annotation_path=(
        PROJECT_ROOT
        / "data"
        / "annotations"
        / "modelA_segmentation_annotations_v3.jsonl"
    ),
    image_root=(
        PROJECT_ROOT
        / "data"
        / "raw"
        / "images"
    ),
    file_names=[
        representative_file_name
    ],
)


# ------------------------------------------------------------
# 2. Dataset 길이 확인
# ------------------------------------------------------------

print(
    "Dataset length:",
    len(single_dataset),
)


# ------------------------------------------------------------
# 3. 실제 sample 하나 꺼내기
#
# 여기서 __getitem__(0)이 실행됨
# ------------------------------------------------------------

dataset_sample = single_dataset[0]


# ------------------------------------------------------------
# 4. 반환 구조 확인
# ------------------------------------------------------------

print()
print(
    "Sample keys:",
    list(dataset_sample.keys()),
)

print(
    "file_name:",
    dataset_sample["file_name"],
)

print()

print(
    "image shape:",
    dataset_sample["image"].shape,
)

print(
    "image dtype:",
    dataset_sample["image"].dtype,
)

print(
    "image min:",
    dataset_sample["image"].min().item(),
)

print(
    "image max:",
    dataset_sample["image"].max().item(),
)

print()

print(
    "seg_target shape:",
    dataset_sample["seg_target"].shape,
)

print(
    "seg_target dtype:",
    dataset_sample["seg_target"].dtype,
)

print(
    "seg_target min:",
    dataset_sample["seg_target"].min().item(),
)

print(
    "seg_target max:",
    dataset_sample["seg_target"].max().item(),
)


# ------------------------------------------------------------
# 5. 각 GT channel pixel 수
# ------------------------------------------------------------

lane_pixels = torch.count_nonzero(
    dataset_sample["seg_target"][0]
).item()

stop_pixels = torch.count_nonzero(
    dataset_sample["seg_target"][1]
).item()

crosswalk_pixels = torch.count_nonzero(
    dataset_sample["seg_target"][2]
).item()


print()
print(
    "lane pixels:",
    lane_pixels,
)

print(
    "stop_line pixels:",
    stop_pixels,
)

print(
    "crosswalk pixels:",
    crosswalk_pixels,
)


# ------------------------------------------------------------
# 6. Contract 검증
# ------------------------------------------------------------

assert len(single_dataset) == 1

assert (
    dataset_sample["file_name"]
    == representative_file_name
)

assert tuple(
    dataset_sample["image"].shape
) == (
    3,
    448,
    768,
)

assert tuple(
    dataset_sample["seg_target"].shape
) == (
    3,
    448,
    768,
)

assert (
    dataset_sample["image"].dtype
    == torch.float32
)

assert (
    dataset_sample["seg_target"].dtype
    == torch.float32
)

assert (
    dataset_sample["image"].shape[-2:]
    ==
    dataset_sample["seg_target"].shape[-2:]
)


# ------------------------------------------------------------
# 7. 우리가 앞에서 직접 검증한 대표 GT와 비교
# ------------------------------------------------------------

assert lane_pixels == 487
assert stop_pixels == 158
assert crosswalk_pixels == 4431


print()
print(
    "SegmentationDataset Single Sample: PASS"
)

Dataset length: 1

Sample keys: ['image', 'seg_target', 'file_name']
file_name: 13886867.jpg

image shape: torch.Size([3, 448, 768])
image dtype: torch.float32
image min: 0.0
image max: 1.0

seg_target shape: torch.Size([3, 448, 768])
seg_target dtype: torch.float32
seg_target min: 0.0
seg_target max: 1.0

lane pixels: 487
stop_line pixels: 158
crosswalk pixels: 4431

SegmentationDataset Single Sample: PASS


In [16]:
# ============================================================
# SegmentationDataset → DataLoader Batch Test
# ============================================================

from torch.utils.data import DataLoader


# ------------------------------------------------------------
# 1. 테스트용 이미지 4장 선택
#
# 대표 이미지 13886867.jpg는 반드시 포함하고
# 나머지 3장은 Dataset에서 가져옴
# ------------------------------------------------------------

test_file_names = [
    representative_file_name
]

for file_name in annotation_groups.keys():

    if file_name == representative_file_name:
        continue

    test_file_names.append(
        file_name
    )

    if len(test_file_names) == 4:
        break


print(
    "Test file names:",
    test_file_names,
)


# ------------------------------------------------------------
# 2. 4장짜리 작은 Dataset 생성
# ------------------------------------------------------------

batch_test_dataset = SegmentationDataset(
    annotation_path=annotation_path,
    image_root=(
        PROJECT_ROOT
        / "data"
        / "raw"
        / "images"
    ),
    file_names=test_file_names,
)


print(
    "Dataset length:",
    len(batch_test_dataset),
)


# ------------------------------------------------------------
# 3. DataLoader 생성
#
# batch_size=4
# → 이미지 4장을 한 번에 묶음
#
# num_workers=0
# → 첫 검증에서는 가장 단순하고
#   디버깅하기 쉬운 설정 사용
# ------------------------------------------------------------

batch_test_loader = DataLoader(
    batch_test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)


# ------------------------------------------------------------
# 4. Batch 하나 꺼내기
# ------------------------------------------------------------

batch = next(
    iter(batch_test_loader)
)


# ------------------------------------------------------------
# 5. 결과 확인
# ------------------------------------------------------------

print()
print("=" * 60)

print(
    "Batch keys:",
    list(batch.keys()),
)

print()

print(
    "file_names:",
    batch["file_name"],
)

print()

print(
    "image shape:",
    batch["image"].shape,
)

print(
    "image dtype:",
    batch["image"].dtype,
)

print()

print(
    "seg_target shape:",
    batch["seg_target"].shape,
)

print(
    "seg_target dtype:",
    batch["seg_target"].dtype,
)


# ------------------------------------------------------------
# 6. 기본 안전성 검사
# ------------------------------------------------------------

image_shape_ok = (
    tuple(batch["image"].shape)
    == (4, 3, 448, 768)
)

target_shape_ok = (
    tuple(batch["seg_target"].shape)
    == (4, 3, 448, 768)
)

image_finite = (
    torch.isfinite(
        batch["image"]
    )
    .all()
    .item()
)

target_finite = (
    torch.isfinite(
        batch["seg_target"]
    )
    .all()
    .item()
)

spatial_match = (
    batch["image"].shape[-2:]
    ==
    batch["seg_target"].shape[-2:]
)


print()
print("=" * 60)

print(
    "Image batch shape PASS:",
    image_shape_ok,
)

print(
    "Target batch shape PASS:",
    target_shape_ok,
)

print(
    "Image finite PASS:",
    image_finite,
)

print(
    "Target finite PASS:",
    target_finite,
)

print(
    "Spatial match PASS:",
    spatial_match,
)


assert image_shape_ok
assert target_shape_ok
assert image_finite
assert target_finite
assert spatial_match


print()
print(
    "Segmentation DataLoader Batch: PASS"
)

Test file names: ['13886867.jpg', '13883084.jpg', '13883085.jpg', '13883471.jpg']
Dataset length: 4

Batch keys: ['image', 'seg_target', 'file_name']

file_names: ['13886867.jpg', '13883084.jpg', '13883085.jpg', '13883471.jpg']

image shape: torch.Size([4, 3, 448, 768])
image dtype: torch.float32

seg_target shape: torch.Size([4, 3, 448, 768])
seg_target dtype: torch.float32

Image batch shape PASS: True
Target batch shape PASS: True
Image finite PASS: True
Target finite PASS: True
Spatial match PASS: True

Segmentation DataLoader Batch: PASS


# 8. 다음 작업 순서

현재 기준:

```text
GT Rasterization                         ✅
Segmentation Head                        ✅
src/segmentation/head.py                ✅
Segmentation Loss                        ✅
src/segmentation/loss.py                ✅

Dataset / DataLoader                     ← 다음
Prediction ↔ 실제 GT Loss 검증
Shared Encoder 실제 연결
Segmentation Training
Validation / Metric
Prediction QC
Multi-task 통합
```

## Git 체크포인트

Head baseline은 이미 별도 commit/push 체크포인트를 만들었다.

Loss 작업도 `src/segmentation/loss.py` 회귀 검증까지 PASS한 상태이므로,
Notebook 저장 후 Loss 관련 변경을 별도 commit/push로 남기는 것이 좋다.

예시 Summary:

```text
feat: add baseline segmentation loss
```

Loss 커밋이 끝난 뒤 Dataset / DataLoader 작업을 시작한다.